In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

def adaptive_sampling(x_orig: np.ndarray, y_orig: np.ndarray, min_points: int = 10) -> tuple[np.ndarray, np.ndarray]:
    """
    Performs adaptive sampling on a polyline by adjusting the step size 
    based on the average local curvature (directional change).

    Args:
        x_orig: Original X coordinates (NumPy array).
        y_orig: Original Y coordinates (NumPy array).
        min_points: Minimum number of points to ensure the reduction doesn't over-simplify.

    Returns:
        A tuple containing the reduced X and Y coordinates (NumPy arrays).
    """
    
    N = len(x_orig)
    if N != len(y_orig) or N < 2:
        print("Error: Input arrays must be of the same length and contain at least 2 points.")
        return x_orig, y_orig

    if N <= min_points:
        print("Warning: Input already below minimum point threshold. Returning original data.")
        return x_orig, y_orig

    # --- 1. Calculate directional change (Curvature Proxy) ---
    
    # We look at the segment starting at P_i and ending at P_i+1.
    # We calculate the change in direction at P_i+1 by comparing (P_i -> P_i+1) 
    # with (P_i+1 -> P_i+2).
    
    # curvature_changes[i] will store the curvature measure at point P_i+1.
    # Its length is N - 2.
    curvature_changes = np.zeros(N - 2)
    
    for i in range(0, N - 2):
        # P_prev = (i), P_curr = (i+1), P_next = (i+2)
        p_prev = (x_orig[i], y_orig[i])
        p_curr = (x_orig[i+1], y_orig[i+1])
        p_next = (x_orig[i+2], y_orig[i+2])
        
        # Angle calculation: atan2(dy, dx)
        angle1 = math.atan2(p_curr[1] - p_prev[1], p_curr[0] - p_prev[0])
        angle2 = math.atan2(p_next[1] - p_curr[1], p_next[0] - p_curr[0])
        
        # Absolute change in angle (Curvature proxy)
        angle_diff = abs(angle2 - angle1)
        
        # We store the curvature change corresponding to the segment starting at i+1
        curvature_changes[i] = angle_diff

    # --- 2. Adaptive Sampling Traversal ---
    
    reduced_x = [x_orig[0]]
    reduced_y = [y_orig[0]]
    
    current_index = 0
    
    # Loop until we are within the final point
    while current_index < N - 1:
        
        remaining_points = N - 1 - current_index
        
        # Handle the final segment: if only 1 or 2 points remain, take the jump of 1 or 2
        if remaining_points <= 1:
            next_index = N - 1
            step = remaining_points
        else:
            # Find the curvature profile for the remaining path.
            # The curvature segment relevant to current_index is indexed from 0 
            # (if current_index=0) up to N - 3 - current_index.
            
            # The number of available curvature measurements is: 
            # Number of points left - 2 = (N - 1 - current_index) - 1 = N - 2 - current_index
            
            # Get the slice of curvature changes for the remaining path
            # Start index: current_index
            # End index: N - 2
            remaining_curvatures = curvature_changes[current_index : N - 2]
            
            # Calculate the average required spacing factor for the remaining path
            # (The factor is inversely proportional to curvature)
            
            # Use a small epsilon to prevent division by zero (for perfectly straight sections)
            epsilon = 1e-6
            avg_spacing_factor = 1.0 / (np.mean(remaining_curvatures) + epsilon)
            
            # Determine the step size:
            # The step size is proportional to the average spacing factor, 
            # but cannot exceed the total remaining points.
            
            # We normalize the average factor to create a practical step size estimate.
            # The factor 100 is a tunable constant that controls the maximum potential step size.
            estimated_step = int(min(remaining_points, max(1, int(avg_spacing_factor * 50))))
            
            # Ensure the step size is at least 1, and respects the physical limit.
            step = max(1, min(estimated_step, remaining_points))

        # Advance the index
        next_index = current_index + step
        
        # Always make sure the final point is included, even if the loop structure misses it slightly
        if next_index >= N - 1:
            next_index = N - 1

        # Add the new point
        reduced_x.append(x_orig[next_index])
        reduced_y.append(y_orig[next_index])
        
        # Move to the new position
        current_index = next_index

    return np.array(reduced_x), np.array(reduced_y)



# --- Example Usage ---
def plot_line(x_orig, y_orig, x_red, y_red, title):
    plt.figure(figsize=(10, 5))
    plt.plot(x_orig, y_orig, label="Original Data", alpha=0.6)
    plt.plot(x_red, y_red, 'r-', linewidth=1, label="Reduced Line", alpha=0.9)
    plt.plot(x_red, y_red, 'ro', markersize=3, label="Key Points")
    plt.title(title)
    plt.xlabel("X Coordinate")
    plt.ylabel("Y Coordinate")
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)
        
# 1. Example Data: A highly curved "S" shape
N_points = 100
t = np.linspace(0, 4 * np.pi, N_points)
x_s = np.sin(t) * 2 + 1
y_s = np.cos(t) * 2 + 1

print("--- Testing S-Curve (High Curvature) ---")
reduced_x_s, reduced_y_s = adaptive_sampling(x_s, y_s)
print(f"Original Points: {N_points}")
print(f"Reduced Points: {len(reduced_x_s)}")
plot_line(x_s, y_s, reduced_x_s, reduced_y_s, title="Original S-Curve")


# 2. Example Data: A mostly straight line with a sharp corner
N_points = 100
x_corner = np.concatenate([
    np.linspace(0, 5, 50),
    np.linspace(5, 10, 50)
])
y_corner = np.concatenate([
    np.full(50, 2.0),
    np.full(50, 2.0) + (np.linspace(0, 5, 50) * 0.1)
])

print("\n--- Testing L-Corner Line (Sharp Change) ---")
reduced_x_c, reduced_y_c = adaptive_sampling(x_corner, y_corner)
print(f"Original Points: {N_points}")
print(f"Reduced Points: {len(reduced_x_c)}")
plot_line(x_corner, y_corner, reduced_x_c, reduced_y_c, title="Original Corner")


# 3. Example Data: A straight line (Low Curvature)
N_points = 100
x_straight = np.linspace(0, 10, N_points)
y_straight = np.full(N_points, 5.0)

print("\n--- Testing Straight Line (Low Curvature) ---")
reduced_x_s, reduced_y_s = adaptive_sampling(x_straight, y_straight)
print(f"Original Points: {N_points}")
print(f"Reduced Points: {len(reduced_x_s)}")
plot_line(x_straight, y_straight, reduced_x_s, reduced_y_s,title="Original Straight")



